# Chapter 18b — Implementing Muon

> Course: **llm.c — Zero to Hero**, companion to Chapter 18 (GPU AdamW).
> Builds on: **Chapter 18a** (the math: momentum → orthogonalize → Newton-Schulz), Chapter 8 (AdamW).
>
> Audience: **freshman** — comfortable with numpy arrays and `for` loops. We turn the math from
> 18a into a working optimizer, line by line.

In 18a you learned *what* Muon does. Now we **build it** and **run it**. By the end you will have:

- A complete `newton_schulz` + `muon_update` in plain numpy (the same math as `KellerJordan/Muon`).
- The **hybrid router** that sends 2D hidden weights to Muon and everything else to AdamW.
- A reproducible demo where Muon **out-converges SGD-momentum** on an ill-conditioned problem.
- A translation bridge from our numpy code to the real PyTorch (bf16) reference.

### Learning objectives

- Implement `newton_schulz` and explain the `transpose` / `normalize` / 5-iteration structure.
- Implement `muon_update` with momentum, the nesterov look-ahead, and the `sqrt(max(1, rows/cols))` scale.
- Write a parameter **router** that picks Muon vs AdamW per tensor by shape/role.
- Demonstrate and interpret Muon's convergence advantage over plain SGD-momentum.


## 1. Newton-Schulz, line by line

This is the heart of Muon — the orthogonalizer from 18a. Read each line against the comment.


In [ ]:
import numpy as np

# Quintic coefficients from KellerJordan/Muon — tuned for speed, not exact convergence.
A_NS, B_NS, C_NS = 3.4445, -4.7750, 2.0315

def newton_schulz(G, steps=5):
    X = G / (np.linalg.norm(G) + 1e-7)        # 1. normalize -> all singular values land in (0, 1]
    transposed = X.shape[-2] > X.shape[-1]    # 2. iterate on the "wide" side (fewer rows) for stability
    if transposed:
        X = X.T
    for _ in range(steps):                    # 3. repeat the quintic 5x
        A = X @ X.T                           #    the ONE expensive op (symmetric) -> Ch18c speeds this up
        B = B_NS * A + C_NS * (A @ A)         #    build b*A + c*A^2
        X = A_NS * X + B @ X                  #    X <- a*X + (b*A + c*A^2) @ X   == apply p() to singular values
    if transposed:
        X = X.T                               # 4. undo the transpose so the shape matches G
    return X

# sanity: orthogonalized output has singular values squashed into a band near 1
rng = np.random.default_rng(0)
G = rng.standard_normal((8, 6))
s_in  = np.linalg.svd(G, compute_uv=False)
s_out = np.linalg.svd(newton_schulz(G), compute_uv=False)
print("spread before:", round(s_in.max()/s_in.min(), 2), "x")
print("spread after: ", round(s_out.max()/s_out.min(), 2), "x  (flattened, no SVD used)")
assert s_out.max()/s_out.min() < s_in.max()/s_in.min(), "NS should flatten the spectrum"
print("PASS")


**Watch the two `X.T` guards.** When a matrix has more rows than columns (`rows > cols`),
`X @ X.T` would be the *big* `rows × rows` matrix. Transposing first makes the inner product the
*small* `cols × cols` one — cheaper and numerically nicer. We flip back at the end so the returned
shape matches the input. This is a free speed win and it's exactly what the reference does.


## 2. `muon_update`: momentum + nesterov + orthogonalize + scale

Now wrap Newton-Schulz with the momentum machinery from 18a. One function returns the **update
direction** for a single 2D weight; the caller multiplies by `-lr` and applies it.


In [ ]:
def muon_update(grad, momentum, beta=0.95, ns_steps=5, nesterov=True):
    # 1. momentum buffer: exponential moving average of the gradient (updated in place)
    momentum[:] = beta * momentum + (1 - beta) * grad
    # 2. nesterov look-ahead: lerp once more toward the fresh gradient
    look = beta * momentum + (1 - beta) * grad if nesterov else momentum
    # 3. orthogonalize the (look-ahead) momentum
    o = newton_schulz(look, steps=ns_steps)
    # 4. shape scale so the effective step size is consistent across layer shapes
    o = o * (max(1, o.shape[-2] / o.shape[-1]) ** 0.5)
    return o

W = rng.standard_normal((128, 64))
M = np.zeros_like(W)
g = rng.standard_normal((128, 64))
d = muon_update(g, M)
print("direction shape:", d.shape)
print("its singular values ~1 (flat):", np.round(np.linalg.svd(d, compute_uv=False)[:4], 3), "...")
print("shape-scale factor sqrt(max(1, 128/64)) =", round(max(1, 128/64) ** 0.5, 4))


## 3. The hybrid router: Muon for matrices, AdamW for the rest

From 18a: Muon is **only** for 2D hidden weight matrices. Embeddings, the output head, and all 1D
tensors (LayerNorm gains, biases) use **AdamW**. Real training keeps both optimizers running side
by side, each owning a subset of parameters. Here is a minimal AdamW step (Chapter 8 math) and the
router that decides who handles what.


In [ ]:
def adamw_update(grad, m, v, t, beta1=0.9, beta2=0.999, eps=1e-8):
    # standard AdamW direction (Chapter 8); caller multiplies by -lr
    m[:] = beta1 * m + (1 - beta1) * grad
    v[:] = beta2 * v + (1 - beta2) * grad * grad
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    return m_hat / (np.sqrt(v_hat) + eps)

def use_muon(name, param):
    # Muon iff it's a 2D *hidden* weight. Embeddings / output head / 1D tensors -> AdamW.
    if param.ndim < 2:
        return False
    if name in ("embedding", "lm_head"):   # lookup tables, not balanced linear maps
        return False
    return True

# a tiny "model": a dict of named parameters of mixed shapes/roles
params = {
    "mlp_w":      rng.standard_normal((64, 64)),   # hidden weight   -> Muon
    "attn_w":     rng.standard_normal((64, 48)),   # hidden weight   -> Muon
    "embedding":  rng.standard_normal((100, 64)),  # lookup table    -> AdamW
    "lm_head":    rng.standard_normal((100, 64)),  # output head     -> AdamW
    "ln_gain":    rng.standard_normal(64),         # 1D LayerNorm    -> AdamW
    "bias":       rng.standard_normal(64),         # 1D bias         -> AdamW
}
for name, p in params.items():
    print(f"{name:10s} shape={str(p.shape):10s} -> {'Muon ' if use_muon(name, p) else 'AdamW'}")


```mermaid
flowchart LR
  subgraph MUON["Muon"]
    mw["mlp_w"]
    aw["attn_w"]
  end
  subgraph ADAMW["AdamW"]
    em["embedding"]
    lm["lm_head"]
    lg["ln_gain"]
    bi["bias"]
  end
  step["one optimizer.step()"] --> MUON
  step --> ADAMW
```

The router runs **once at setup**: it splits parameters into two groups, and each `step()` updates
both groups. This is exactly the parameter-group pattern in `KellerJordan/Muon`'s README
(`use_muon=True` vs `use_muon=False`).


## 4. Demo — Muon out-converges SGD-momentum

Orthogonalization's payoff shows up on an **ill-conditioned** problem: one where a few directions
have huge gradients and others are tiny (recall the lopsided spectrum from 18a). Plain
SGD-momentum crawls along the quiet directions; Muon equalizes them and makes uniform progress.

We fit a weight matrix `W` to minimize `0.5·‖W X − Y‖²` with deliberately ill-conditioned inputs
`X` (condition number ~100). Both optimizers use momentum 0.95; only Muon adds orthogonalization.


In [ ]:
# Reproducible ill-conditioned least-squares for a weight matrix.
rng = np.random.default_rng(0)
d, n = 24, 80
W_true = rng.standard_normal((d, d))
Uc, _ = np.linalg.qr(rng.standard_normal((d, d)))
cond = np.geomspace(1.0, 0.01, d)             # singular values span 100x -> ill-conditioned
X = (Uc * cond) @ rng.standard_normal((d, n))
Y = W_true @ X

def loss(W):  r = W @ X - Y; return 0.5 * np.sum(r * r) / n
def grad(W):  return ((W @ X - Y) @ X.T) / n

def train(kind, steps=80, lr=0.5):
    W = np.zeros((d, d)); M = np.zeros_like(W); hist = [loss(W)]
    for t in range(1, steps + 1):
        g = grad(W)
        if kind == "sgd":
            M = 0.95 * M + g
            W = W - lr * M
        else:  # muon
            d_dir = muon_update(g, M)
            W = W - lr * d_dir
        hist.append(loss(W))
    return hist

h_sgd  = train("sgd",  80, lr=0.6)
h_muon = train("muon", 80, lr=0.5)
for k in (0, 5, 10, 20, 40, 80):
    print(f"step {k:3d}:  SGD-momentum = {h_sgd[k]:8.4f}    Muon = {h_muon[k]:8.4f}")
print(f"\nfinal:  SGD-momentum = {h_sgd[-1]:.4f}   Muon = {h_muon[-1]:.4f}   "
      f"({h_sgd[-1]/h_muon[-1]:.1f}x lower with Muon)")
assert h_muon[-1] < h_sgd[-1], "Muon should reach a lower loss here"
print("PASS")


Plotting both loss curves on a log scale makes the gap obvious — Muon's curve sits below SGD's at
every step:

![Muon vs SGD-momentum loss curves on an ill-conditioned problem](course/figures/fig_18b_convergence.png)

*(Figure generated by `course/build_ch18b_figures.py`, which reruns the exact training above.)*


**Read the result.** Muon's loss is lower at *every* checkpoint and ends ~1.7× lower. The
mechanism is exactly 18a's picture: SGD keeps stepping mostly along the few loud directions, while
Muon's orthogonalized update treats all directions evenly, so the quiet ones stop lagging.

> Note: against **AdamW**, on a small convex toy like this, AdamW's per-coordinate adaptivity is
> also very strong — Muon's documented wins are in real LLM training, and AdamW is Muon's *partner*
> (the hybrid in §3), not its rival on a toy. The clean, mechanism-level comparison is vs SGD.


## 5. Translation Bridge — our numpy vs the real PyTorch

| Our numpy (this chapter) | `KellerJordan/Muon` (`muon.py`, PyTorch) |
|---|---|
| `X / (np.linalg.norm(X) + 1e-7)` | `X / (X.norm(dim=(-2,-1), keepdim=True) + 1e-7)` |
| `X.T` guard when `rows > cols` | `if G.size(-2) > G.size(-1): X = X.mT` |
| `A = X @ X.T; B = b*A + c*(A@A); X = a*X + B@X` | identical, on `X.bfloat16()` |
| `momentum[:] = beta*momentum + (1-beta)*grad` | `momentum.lerp_(grad, 1 - beta)` |
| `look = beta*momentum + (1-beta)*grad` | `update = grad.lerp_(momentum, beta)` (nesterov) |
| `o * max(1, rows/cols)**0.5` | `update *= max(1, update.size(-2)/update.size(-1))**0.5` |
| `use_muon(name, p)` router | parameter groups with `use_muon=True/False` |

The only real difference: the reference casts `X` to **bfloat16** before the iteration (the whole
point of the speed-tuned coefficients — they stay stable in 16-bit), and fuses the momentum with
`lerp_`. The math is what you just wrote.

```python
# The reference zeropower_via_newtonschulz5, verbatim (PyTorch):
def zeropower_via_newtonschulz5(G, steps: int):
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16()
    if G.size(-2) > G.size(-1):
        X = X.mT
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        A = X @ X.mT
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(-2) > G.size(-1):
        X = X.mT
    return X
```


## 6. Exercises

Predict before running. Solutions are collapsed.


### Exercise 1 — the momentum buffer must persist

A classic bug: re-creating the momentum buffer `M` inside the loop instead of reusing it. Modify
`train("muon", ...)` to allocate a **fresh** `M = np.zeros_like(W)` *every step*. **Predict:** does
Muon get better, worse, or unchanged? Why?


In [ ]:
# Your attempt: copy train() but reset M each step inside the loop, compare final loss.
def train_buggy(steps=80, lr=0.5):
    W = np.zeros((d, d)); hist = [loss(W)]
    for t in range(1, steps + 1):
        g = grad(W)
        # TODO: M = np.zeros_like(W)  (reset every step -- the bug)
        # TODO: W = W - lr * muon_update(g, M)
        hist.append(loss(W))
    return hist


<details>
<summary>▶ Show solution</summary>

```python
def train_buggy(steps=80, lr=0.5):
    W = np.zeros((d, d)); hist = [loss(W)]
    for t in range(1, steps + 1):
        g = grad(W)
        M = np.zeros_like(W)            # BUG: momentum forgotten every step
        W = W - lr * muon_update(g, M)
        hist.append(loss(W))
    return hist

print("buggy (no memory):", round(train_buggy()[-1], 4), " vs correct:", round(h_muon[-1], 4))
# Worse: with M reset each step, the momentum EMA never accumulates, so "look" is just a scaled
# gradient. You lose the variance-smoothing of momentum and converge slower / noisier.
```

Momentum is **state** — it must live across steps. This is why the optimizer stores a
`momentum_buffer` per parameter (and why GPU Muon must keep that buffer in device memory).
</details>


### Exercise 2 — does the shape-scale matter?

The scale factor is `sqrt(max(1, rows/cols))`. For the square `24×24` weight in the demo it equals
`1.0`, so it does nothing. Compute the factor for a **non-square** `48×24` weight by hand, then
check with code. **Predict:** dropping the scale acts like multiplying the learning rate by what?


In [ ]:
# Your attempt: print the shape-scale for 24x24 and 48x24.
# TODO: print(max(1, 24/24) ** 0.5)
# TODO: print(max(1, 48/24) ** 0.5)


<details>
<summary>▶ Show solution</summary>

```python
print("scale for 24x24:", round(max(1, 24/24) ** 0.5, 4))   # 1.0   -> dropping it changes nothing
print("scale for 48x24:", round(max(1, 48/24) ** 0.5, 4))   # 1.4142
```

For a square matrix the factor is `1.0`. For `48×24` it is `√2 ≈ 1.414`, so dropping the scale
shrinks the update ~1.4× — equivalent to using a ~1.4× smaller learning rate on that layer. The
shape-scale keeps the **effective step size consistent across differently-shaped layers**, so one
global learning rate works for the whole network.
</details>


### Exercise 3 — extend the router

Suppose your model adds a `conv_w` parameter of shape `(64, 64, 3, 3)` (a 4D conv weight). The
reference flattens 4D weights to 2D (`update.view(len(update), -1)`) and *then* orthogonalizes.
Update `use_muon` and `muon_update` to handle `ndim == 4` by reshaping to `(64, 64*3*3)`. Should a
conv weight go to Muon or AdamW?


<details>
<summary>▶ Show solution</summary>

```python
def use_muon(name, param):
    if param.ndim < 2:                 # 1D -> AdamW
        return False
    if name in ("embedding", "lm_head"):
        return False
    return True                        # 2D and 4D hidden weights -> Muon

def muon_update(grad, momentum, beta=0.95, ns_steps=5, nesterov=True):
    momentum[:] = beta * momentum + (1 - beta) * grad
    look = beta * momentum + (1 - beta) * grad if nesterov else momentum
    if look.ndim == 4:                 # flatten conv (out, in, kh, kw) -> (out, in*kh*kw)
        look = look.reshape(look.shape[0], -1)
    o = newton_schulz(look, steps=ns_steps)
    o = o * (max(1, o.shape[-2] / o.shape[-1]) ** 0.5)
    return o   # caller reshapes back to the param's shape before applying
```

A conv weight is a (reshaped) linear map, so it **goes to Muon** — exactly what the reference does
with `update.view(len(update), -1)`.
</details>


## Further Reading

**Source of truth**

- [`KellerJordan/Muon`](https://github.com/KellerJordan/Muon) (`muon.py`) — the reference `zeropower_via_newtonschulz5`, `muon_update`, and the parameter-group router we mirrored.
- [Keller Jordan — *Muon* write-up](https://kellerjordan.github.io/posts/muon/) — defaults (`momentum=0.95`, `nesterov=True`, `ns_steps=5`) and which parameters Muon owns.

**Going deeper**

- [PyTorch blog — *Using the Muon optimizer with DeepSpeed*](https://pytorch.org/blog/using-muon-optimizer-with-deepspeed/) — distributing Muon's per-matrix orthogonalization across GPUs.
- [NVIDIA NeMo-RL — *Muon optimizer* guide](https://docs.nvidia.com/nemo/rl/latest/guides/muon-optimizer.html) — production integration and hyperparameters.

**Sibling chapters**

- **Chapter 18a** — the math behind every line here.
- **Chapter 18c** — implement Newton-Schulz in CUDA/C++ and make the `X Xᵀ` step fast.


## Recap

- `newton_schulz`: normalize → (transpose if `rows>cols`) → 5× quintic of matmuls → untranspose. The one costly op is `A = X Xᵀ`.
- `muon_update`: momentum EMA → nesterov look-ahead → orthogonalize → `sqrt(max(1, rows/cols))` shape-scale.
- The **hybrid router** sends 2D hidden weights to Muon and embeddings/head/1D tensors to AdamW — both run every step.
- On an ill-conditioned problem, Muon **out-converges SGD-momentum** because orthogonalization unsticks the quiet directions.
- Our numpy maps 1:1 onto the PyTorch reference; the only real change there is the **bf16** cast that the speed-tuned coefficients are designed for.

### What's next

**Chapter 18c — Muon on CUDA/C++.** The `A = X Xᵀ` line runs 5× per matrix per step over thousands
of matrices. We implement Newton-Schulz in CUDA — first a **minimal** correct version, then an
**optimal** one using the flash-muon trick (compute only the upper triangle of the symmetric
`X Xᵀ`) — and benchmark the speedup on the GPU.

When you're ready, say **"proceed to Chapter 18c"**.
